# Multi-Metric Backtest: Late-Day PUT Explosion Prediction

## Objective
Test whether gamma-related metrics (GCI, PGR, GDW, CAR, Charm, Vomma, Zomma) predict dramatic PUT option price explosions during late-day SPX selloffs.

## Pre-Registered Hypotheses
1. **Primary**: GCI > 0.30 predicts PUT % gain > 100% with lift > 1.5x
2. **Primary**: GCI > 0.30 AND PGR < 0.25 outperforms GCI alone
3. **Secondary**: All other metrics and composites (exploratory, stricter threshold)

## Key Methodology
- Measure PUT % gains, NOT SPX point moves
- Use Spearman correlation (fat-tailed returns)
- Apply Benjamini-Hochberg FDR correction
- Use bid price for entry (conservative)
- Run control experiments before trusting results

## Architecture
This notebook uses the modular `src/` package. The code is organized as:
- `src/config.py` - Configuration and thresholds
- `src/data_loader.py` - Data loading utilities
- `src/greeks.py` - Black-Scholes Greeks calculations
- `src/metrics.py` - GCI, PGR, GDW, CAR calculations
- `src/put_tracker.py` - PUT price tracking
- `src/statistics.py` - Lift, permutation, FDR functions
- `src/visualization.py` - Plotting functions
- `src/processor.py` - Main processing orchestration

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path for src imports
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import our modules
from src import Config, BacktestRunner, DataLoader
from src.visualization import Visualizer
from src.statistics import StatisticalAnalyzer

# For interactive exploration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
sns.set_palette('husl')

print(f'Project directory: {PROJECT_ROOT}')
print(f'Data directory: {PROJECT_ROOT / "data"}')


---
## 2. Configuration

In [ ]:
# Initialize configuration
config = Config()

print("Study Configuration:")
print(f"  Period: {config.study_start} to {config.study_end}")
print(f"  Interval: {config.interval_minutes} minutes")
print(f"  Strike range: ±${config.strike_range}")
print(f"  Time horizons: {config.time_horizons} minutes")

print("\nPre-Registered Thresholds:")
print(f"  GCI: > {config.thresholds.gci}")
print(f"  PGR: < {config.thresholds.pgr} (inverted)")
print(f"  CAR net: > {config.thresholds.car_net}")
print(f"  FDR alpha: {config.stats.fdr_alpha}")

print(f"\nData directory: {config.get_data_dir()}")

---
## 3. Check Data Availability

In [ ]:
loader = DataLoader(config)
available_dates = loader.get_available_dates()

print(f"Available trading days: {len(available_dates)}")
if available_dates:
    print(f"Date range: {available_dates[0]} to {available_dates[-1]}")
else:
    print("\nNo data found. Please ensure parquet files are available at:")
    print(f"  {config.local_data_dir}")
    print("\nOr SSH to backfill server:")
    print("  scp root@45.55.51.49:/tmp/backfill_output/*.parquet ./data/")

---
## 4. Run Backtest

Process all available days and calculate metrics + PUT returns.

In [ ]:
# Initialize the backtest runner
runner = BacktestRunner(config)

# Run on all data (or limit for testing)
# Use limit=10 for quick testing, None for full analysis
LIMIT = 10  # Set to None for full analysis

df_all = runner.run(limit=LIMIT, show_progress=True)

print(f"\nProcessed {len(df_all)} intervals across {df_all['date'].nunique()} days")

In [ ]:
# Preview the data
if len(df_all) > 0:
    print("Columns:")
    print(df_all.columns.tolist())
    print("\nSample data:")
    display(df_all.head())
    print("\nMetrics summary:")
    display(df_all[['gci', 'pgr', 'car_net', 'pct_gain_30m']].describe())

---
## 5. Control Experiments

Run control experiments FIRST to validate methodology before trusting any results.

In [ ]:
if len(df_all) > 0:
    controls = runner.run_control_experiments(outcome_threshold=100.0, n_permutations=500)
    
    print("=" * 60)
    print("CONTROL EXPERIMENTS")
    print("=" * 60)
    
    # Placebo test
    print("\n1. PLACEBO TEST (Random Signal)")
    print("-" * 40)
    placebo = controls['placebo']
    print(f"Random signal lift: {placebo['lift']:.2f}")
    print(f"95% CI: {placebo['ci_low']:.2f} - {placebo['ci_high']:.2f}")
    print(f"Expected: {placebo['expected']}")
    if placebo['pass']:
        print("✓ PASS: Random signal has no predictive power")
    else:
        print("⚠ WARNING: Random signal shows unexpected lift")
    
    # Time-shifted test
    if 'time_shifted' in controls:
        print("\n2. TIME-SHIFTED TEST (Future GCI)")
        print("-" * 40)
        ts = controls['time_shifted']
        print(f"Future GCI lift: {ts['lift']:.2f}")
        print(f"Expected: {ts['expected']}")
    
    # Permutation test
    print("\n3. PERMUTATION TEST")
    print("-" * 40)
    perm = controls['permutation']
    print(f"Observed lift: {perm['observed_lift']:.2f}")
    print(f"Null distribution: mean={perm['null_mean']:.2f}, std={perm['null_std']:.2f}")
    print(f"Permutation p-value: {perm['p_value']:.4f}")
    if perm['significant']:
        print("✓ SIGNIFICANT: Observed lift unlikely due to chance")
    else:
        print("NOT SIGNIFICANT: Cannot reject null hypothesis")
else:
    print("No data available for control experiments.")

---
## 6. Univariate Analysis

Test each metric individually across all time horizons.

In [ ]:
if len(df_all) > 0:
    univar_results = runner.run_univariate_analysis(outcome_threshold=100.0)
    
    print("=" * 60)
    print("UNIVARIATE ANALYSIS RESULTS")
    print("=" * 60)
    print("\nResults sorted by lift (highest first):")
    
    cols = ['metric', 'window', 'spearman_r', 'lift', 'ci_low', 'ci_high', 
            'fisher_p', 'p_adjusted', 'significant', 'n_signals']
    display(univar_results[cols])
    
    # Highlight significant results
    sig_results = univar_results[univar_results['significant']]
    print(f"\nSIGNIFICANT RESULTS (FDR < {config.stats.fdr_alpha}):")
    if len(sig_results) > 0:
        display(sig_results[['metric', 'window', 'lift', 'p_adjusted']])
    else:
        print("None - no metrics survive FDR correction")
else:
    print("No data available for analysis.")

---
## 7. Composite Signal Analysis

Test combinations of metrics.

In [ ]:
if len(df_all) > 0:
    comp_results = runner.run_composite_analysis(outcome_threshold=100.0)
    
    print("=" * 60)
    print("COMPOSITE SIGNAL RESULTS")
    print("=" * 60)
    print("\nComparison of signal strategies (30-min PUT gain > 100%):")
    display(comp_results)
    
    # Compare composite vs single
    if len(comp_results) >= 2:
        gci_lift = comp_results[comp_results['signal'] == 'GCI alone']['lift'].values
        composite_lift = comp_results[comp_results['signal'] == 'GCI + Low PGR']['lift'].values
        
        if len(gci_lift) > 0 and len(composite_lift) > 0:
            improvement = composite_lift[0] / gci_lift[0]
            print(f"\nComposite vs GCI alone: {improvement:.2f}x")
            if improvement > 1.2:
                print("✓ FINDING: Composite signal outperforms single metric")
            else:
                print("FINDING: No significant improvement from composite")
else:
    print("No data available for analysis.")

---
## 8. Visualizations

In [ ]:
if len(df_all) > 0 and 'univar_results' in dir() and len(univar_results) > 0:
    viz = Visualizer(config)
    
    # Correlation heatmap
    print("Generating correlation heatmap...")
    heatmap_path = viz.plot_correlation_heatmap(univar_results)
    print(f"Saved: {heatmap_path}")
    
    # Display inline
    from IPython.display import Image, display
    display(Image(filename=str(heatmap_path)))

In [ ]:
if len(df_all) > 0 and 'pct_gain_30m' in df_all.columns:
    # PUT return distribution
    print("Generating PUT return distribution...")
    dist_path = viz.plot_put_return_distribution(
        df_all,
        signal_col='gci',
        threshold=config.thresholds.gci
    )
    print(f"Saved: {dist_path}")
    
    from IPython.display import Image, display
    display(Image(filename=str(dist_path)))

In [ ]:
if 'comp_results' in dir() and len(comp_results) > 0:
    # Composite comparison
    print("Generating composite comparison...")
    comp_path = viz.plot_composite_comparison(comp_results)
    print(f"Saved: {comp_path}")
    
    from IPython.display import Image, display
    display(Image(filename=str(comp_path)))

---
## 9. Results Summary & Decision Framework

In [ ]:
runner.print_summary()

---
## 10. Save Results

In [ ]:
if len(df_all) > 0:
    saved_files = runner.save_results()
    
    print("Saved files:")
    for f in saved_files:
        print(f"  - {f}")
else:
    print("No data to save.")

---
## 11. Interactive Exploration

Use this section for ad-hoc analysis.

In [ ]:
# Example: Look at a specific day
if len(df_all) > 0:
    sample_date = df_all['date'].iloc[0]
    df_day = df_all[df_all['date'] == sample_date]
    
    print(f"\nSample day: {sample_date}")
    print(f"Intervals: {len(df_day)}")
    
    # Plot metrics for this day
    if len(df_day) > 5:
        viz.plot_metric_timeseries(df_day, sample_date, metrics=['gci', 'pgr', 'car_net'])
        print(f"Saved: {config.results_dir / f'metrics_timeseries_{sample_date}.png'}")

In [ ]:
# Example: Custom correlation analysis
if len(df_all) > 0:
    stats_analyzer = StatisticalAnalyzer(config)
    
    # Test custom metric combination
    custom_signal = (df_all['gci'] > 0.25) & (df_all['car_net'] > 1.5)
    outcome = df_all['pct_gain_30m'] > 100
    
    valid_mask = custom_signal.notna() & outcome.notna()
    
    if valid_mask.sum() > 30:
        result = stats_analyzer.calculate_lift(
            custom_signal[valid_mask], 
            outcome[valid_mask]
        )
        print("\nCustom signal: GCI > 0.25 AND CAR > 1.5")
        print(f"Lift: {result.lift:.2f} (95% CI: {result.ci_low:.2f} - {result.ci_high:.2f})")
        print(f"p-value: {result.p_value:.4f}")
        print(f"Signal frequency: {custom_signal.mean():.1%}")